**Proyecto II de Programación para Estadística ll**

**Tema:** La evolución del gasto corriente en salud (% del PIB), la pobreza y la incidencia de las enfermedades transmisibles y su variación entre países con diferentes desarrollos económicos durante la última década

**Profesor:** Michael Sanchez Soto

**Estudiantes:** Cyra Coto, Gustavo Jiménez y Abigail Murillo

**Pregunta**

¿Cómo ha evolucionado el gasto corriente en salud (% del PIB), la pobreza y la incidencia de las enfermedades transmisibles durante la última década y de qué manera varía esta relación entre países con diferentes niveles de desarrollo económico?

# 1. CARGA DE LAS BASES DE DATOS

In [262]:
#Importación de las librerías y carga de los datas frames crudos
import pandas as pd
import numpy as np
import json
from google.colab import files

# -----------------------
# 1. CARGA DE ARCHIVOS RAW
# -----------------------

print("SUBE ARCHIVO DENGUE 1")
uploaded_dengue = files.upload()
df_dengue = pd.read_excel(list(uploaded_dengue.keys())[0])

print("SUBE ARCHIVO HIV")
uploaded_hiv = files.upload()
df_HIV = pd.read_excel(
    list(uploaded_hiv.keys())[0],
    sheet_name="HIV-Test-&-Treat_ByYear",
    engine="openpyxl",
    skiprows=6
)

print("SUBE ARCHIVO INDICE DE POBREZA")
uploaded_pobreza = files.upload()
df_indice_pobreza = pd.read_csv(list(uploaded_pobreza.keys())[0])

print("SUBE ARCHIVO TUBERCULOSIS")
uploaded_tb = files.upload()
df_tuberculosis = pd.read_csv(list(uploaded_tb.keys())[0])

# -----------------------
# 2. CARGA DEL JSON
# -----------------------

print("SUBE ARCHIVO JSON DE GASTO EN SALUD")
uploaded_json = files.upload()
json_filename = list(uploaded_json.keys())[0]
print("JSON cargado:", json_filename)

# Leer JSON correctamente
with open(json_filename, "r", encoding="utf-8") as f:
    data_json = json.load(f)

# Normalizar JSON en DataFrame
df_gasto_salud = pd.json_normalize(data_json)

# -----------------------
# 3. SEGUNDO DATA FRAME DEL DENGUE
# -----------------------

print("SUBE ARCHIVO DENGUE 2")
uploaded_dengue2 = files.upload()
df_dengue2 = pd.read_csv(list(uploaded_dengue2.keys())[0])


SUBE ARCHIVO DENGUE 1


Saving dengue-global-data-2025-11-12.xlsx to dengue-global-data-2025-11-12 (6).xlsx
SUBE ARCHIVO HIV


Saving HIV_estimates_from_1990-to-present.xlsx to HIV_estimates_from_1990-to-present (6).xlsx
SUBE ARCHIVO INDICE DE POBREZA


Saving indice de pobreza global.csv to indice de pobreza global (6).csv
SUBE ARCHIVO TUBERCULOSIS


Saving number-of-tuberculosis-cases.csv to number-of-tuberculosis-cases (4).csv
SUBE ARCHIVO JSON DE GASTO EN SALUD


Saving Gasto_corriente_en_salud_PIB(1).json to Gasto_corriente_en_salud_PIB(1) (6).json
JSON cargado: Gasto_corriente_en_salud_PIB(1) (6).json
SUBE ARCHIVO DENGUE 2


Saving National_extract_V1_3.csv to National_extract_V1_3 (4).csv


# 2. PREPARACIÓN DE LA BASE DEL DENGUE

In [263]:
df_dengue.columns #exploracion de las columnas del data frame del dengue

Index(['date', 'date_lab', 'who_region', 'who_region_long', 'country', 'iso3',
       'cases', 'confirmed_cases', 'severe_cases', 'deaths', 'cfr', 'prop_sev',
       'cfr_ci_lower', 'cfr_ci_upper', 'prop_sev_ci_lower',
       'prop_sev_ci_upper'],
      dtype='object')

In [264]:
display(df_dengue) #exploracion del df crudo

,date,date_lab,who_region,who_region_long,country,iso3,cases,confirmed_cases,severe_cases,deaths,cfr,prop_sev,cfr_ci_lower,cfr_ci_upper,prop_sev_ci_lower,prop_sev_ci_upper
0,2010-01-01,Jan 2010,SEAR,South-East Asia Region,Thailand,THA,1496.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-01,Feb 2010,SEAR,South-East Asia Region,Thailand,THA,1528.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-03-01,Mar 2010,SEAR,South-East Asia Region,Thailand,THA,2020.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-04-01,Apr 2010,SEAR,South-East Asia Region,Thailand,THA,1868.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-05-01,May 2010,SEAR,South-East Asia Region,Thailand,THA,3666.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9900,2025-09-01,Sep 2025,WPR,Western Pacific Region,Tokelau,TKL,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9901,2025-09-01,Sep 2025,WPR,Western Pacific Region,Tonga,TON,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9902,2025-09-01,Sep 2025,WPR,Western Pacific Region,Tuvalu,TUV,15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9903,2025-09-01,Sep 2025,WPR,Western Pacific Region,Vanuatu,VUT,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [265]:
df_dengue.isna().sum() #La base tiene varios varios NA's

,0
date,0
date_lab,0
who_region,0
who_region_long,0
country,0
iso3,0
cases,1328
confirmed_cases,3900
severe_cases,4299
deaths,2222


In [266]:
df_dengue.dtypes #Vemos que que las variables están en el formato correcto, además todos los valores faltantes son númericos, por lo que tenemos la posibilidad de sustituirlos con el valor de la mediana


,0
date,datetime64[ns]
date_lab,object
who_region,object
who_region_long,object
country,object
iso3,object
cases,float64
confirmed_cases,float64
severe_cases,float64
deaths,float64


In [267]:
df_dengue["cases"] = df_dengue["cases"].fillna("No indica")
df_dengue["confirmed_cases"] = df_dengue["confirmed_cases"].fillna("No indica") #LLenamos los valores faltantes con 0 ya que no hay registros
df_dengue["severe_cases"] = df_dengue["severe_cases"].fillna("No indica")
df_dengue["deaths"] = df_dengue["deaths"].fillna("No indica")
df_dengue["cfr"] = df_dengue["cfr"].fillna("Desconocido")
df_dengue["cfr_ci_upper"] = df_dengue["cfr_ci_upper"].fillna("Desconocido") #No sabemos hasta que número podría llegar
df_dengue["cfr_ci_lower"] = df_dengue["cfr_ci_lower"].fillna("Desconocido")
df_dengue["prop_sev"] = df_dengue["prop_sev"].fillna("Desconocido")
df_dengue["prop_sev_ci_lower"] = df_dengue["prop_sev_ci_lower"].fillna("Desconocido")
df_dengue["prop_sev_ci_upper"] = df_dengue["prop_sev_ci_upper"].fillna("Desconocido") #No sabemos hasta que número podría llegar
df_dengue["prop_sev_ci_lower"] = df_dengue["prop_sev_ci_lower"].fillna("Desconocido")



In [268]:
df_dengue.isna().sum() # Hemos limpiado los NA's

,0
date,0
date_lab,0
who_region,0
who_region_long,0
country,0
iso3,0
cases,0
confirmed_cases,0
severe_cases,0
deaths,0


In [270]:
df_dengue.duplicated().sum() #No posee duplicados

np.int64(0)

In [269]:

#Asegúrate de que 'date' sea datetime
df_dengue['date'] = pd.to_datetime(df_dengue['date'])

# Crear columna 'year'
df_dengue['year'] = df_dengue['date'].dt.year


# Convertir 'cases' a numérico, forzando errores a NaN
df_dengue['cases'] = pd.to_numeric(df_dengue['cases'], errors='coerce')

# Agrupar por país y año
df_dengue_final = df_dengue.groupby(['country', 'year'], as_index=False)['cases'].sum()

df_dengue_final.to_excel("dengue.xlsx", index=False)


## 2.2 SEGUNDA BASE DEL DENGUE

In [271]:
# --- PASO 1: quedarte solo con columnas importantes ---
df_dengue2x = df_dengue2[["adm_0_name", "ISO_A0", "Year", "dengue_total"]]

# --- PASO 2: convertir 'dengue_total' a numérico por si viene como string ---
df_dengue2["dengue_total"] = pd.to_numeric(df_dengue2["dengue_total"], errors="coerce")

# --- PASO 3: agrupar por país y año ---
df_dengue2_final = (
    df_dengue2.groupby(["adm_0_name", "ISO_A0", "Year"], as_index=False)
      .agg(total_cases=("dengue_total", "sum"))
)


df_dengue2_final.rename(columns={
    'adm_0_name': 'country',
    'ISO_A0': 'code',
    'Year': 'year',
    'total_cases': 'dengue_cases'
}, inplace=True)

# Verificar
print(df_dengue2_final.head())



# Resultado final
df_dengue2_final


       country code  year  dengue_cases
0  AFGHANISTAN  AFG  2021         734.0
1  AFGHANISTAN  AFG  2022        1341.0
2  AFGHANISTAN  AFG  2023        1481.0
3  AFGHANISTAN  AFG  2024        4850.0
4  AFGHANISTAN  AFG  2025         115.0


,country,code,year,dengue_cases
0,AFGHANISTAN,AFG,2021,734.0
1,AFGHANISTAN,AFG,2022,1341.0
2,AFGHANISTAN,AFG,2023,1481.0
3,AFGHANISTAN,AFG,2024,4850.0
4,AFGHANISTAN,AFG,2025,115.0
...,...,...,...,...
4328,YEMEN,YEM,2020,62028.0
4329,YEMEN,YEM,2021,7324.0
4330,YEMEN,YEM,2022,24545.0
4331,YEMEN,YEM,2023,21302.0


# 3. PREPARACION DE LA BASE DE LA TUBERCULOSIS

In [272]:
display(df_tuberculosis)

,Entity,Code,Year,Estimated number of new cases of all forms of tuberculosis
0,Afghanistan,AFG,2000,38000
1,Afghanistan,AFG,2001,38000
2,Afghanistan,AFG,2002,40000
3,Afghanistan,AFG,2003,43000
4,Afghanistan,AFG,2004,44000
...,...,...,...,...
5376,Zimbabwe,ZWE,2019,30000
5377,Zimbabwe,ZWE,2020,29000
5378,Zimbabwe,ZWE,2021,31000
5379,Zimbabwe,ZWE,2022,34000


In [273]:
df_tuberculosis.isna().sum() #La base posee varios NA's

,0
Entity,0
Code,240
Year,0
Estimated number of new cases of all forms of tuberculosis,0


In [274]:
df_tuberculosis.dtypes # Todas las variables son del tipo correcto de datos

,0
Entity,object
Code,object
Year,int64
Estimated number of new cases of all forms of tuberculosis,int64


In [275]:
df_tuberculosis["Code"] = df_tuberculosis["Code"].fillna("No especifica")

# 4. PREPARACION DE LA BASE GASTOS EN SALUD json

In [276]:
#Esta base es la de tipo json. requiere un tratamiento especial
#Quitar las primeras filas que no aportan informacion
df_temp = df_gasto_salud.copy()
df_temp = df_temp.iloc[2:].reset_index(drop=True)

#Usar la primera fila como encabezado real
#La nueva fila 0 ahora tiene los nombres reales de columnas (Country Name, Country Code, Indicator Name, 2000, 2001, ...)
new_header = df_temp.iloc[0]
df_temp = df_temp.iloc[1:].reset_index(drop=True)
df_temp.columns = new_header

#Eliminar columnas 'Unnamed' y vacías que hay varias
df_temp = df_temp.loc[:, ~df_temp.columns.astype(str).str.contains(r'^Unnamed', na=False)]
df_temp = df_temp[[c for c in df_temp.columns if str(c).strip() != '' and c is not None]]

#Pasar a formato largo (tidy) year–value o hacer la transposición
#nombres de columnas
rename_map = {
    'Country Name': 'country_name',
    'Country Code': 'country_code',
    'Indicator Name': 'indicator_name',
    'Indicator Code': 'indicator_code'  # por si existiera
}
df_temp = df_temp.rename(columns={k: v for k, v in rename_map.items() if k in df_temp.columns})

#Detectar columnas de año: están como '2000.0', '2001.0',...
def es_col_anio(col):
    s = str(col).strip()
    s = s[:-2] if s.endswith('.0') else s
    return s.isdigit() and len(s) == 4

year_cols = [c for c in df_temp.columns if es_col_anio(c)]
if not year_cols:
    raise ValueError("No se detectaron columnas de año. Revisa los nombres (p.ej. '2000.0').")

#Construir id_vars presentes
id_vars = [c for c in ['country_name', 'country_code', 'indicator_name', 'indicator_code'] if c in df_temp.columns]

df_largo = df_temp.melt(
    id_vars=id_vars,
    value_vars=year_cols,
    var_name='year_raw',
    value_name='value'
)

#Convertir 'year_raw' a 'year' numérico limpio
def limpiar_year(y):
    s = str(y).strip()
    s = s[:-2] if s.endswith('.0') else s
    return pd.to_numeric(s, errors='coerce')

df_largo['year'] = df_largo['year_raw'].apply(limpiar_year)
df_largo.drop(columns=['year_raw'], inplace=True)

#Tipos y limpieza básica
df_largo['value'] = pd.to_numeric(df_largo['value'], errors='coerce')
df_largo = df_largo.dropna(subset=['year', 'value'])

#Quitar filas sin país o con 'None'
if 'country_name' in df_largo.columns:
    df_largo = df_largo[df_largo['country_name'].notna() & (df_largo['country_name'].astype(str).str.strip().str.lower() != 'none')]

#Filtrar el indicador que interesa
target_names = [
    "Gasto corriente en salud (% del PIB)",
    "Current health expenditure (% of GDP)"
]
df_temporal2 = df_largo[df_largo['indicator_name'].isin(target_names)].copy()

#dejar sólo países con código ISO3 (evitar agregados/regiones)
if 'country_code' in df_temporal2.columns:
    df_temporal2 = df_temporal2[df_temporal2['country_code'].astype(str).str.len() == 3]

#Orden final
df_temporal2 = df_temporal2.sort_values(by=['country_name', 'year']).reset_index(drop=True)

out_min = df_temporal2.rename(columns={'country_name': 'pais'})[['pais', 'year', 'value']]

#metadatos útiles para merge:
out_full = df_temporal2.rename(columns={'country_name': 'pais'})[['pais', 'country_code', 'indicator_name', 'year', 'value']]

# Reordenar columnas: country_name, country_code, indicator_name, year, value
cols_order = ['country_name', 'country_code', 'indicator_name', 'year', 'value']
df_gastos_en_salud_final = df_temporal2[cols_order].copy()

#asegurar tipos
df_gastos_en_salud_final['year'] = pd.to_numeric(df_gastos_en_salud_final['year'], errors='coerce').astype('Int64')
df_gastos_en_salud_final['value'] = pd.to_numeric(df_gastos_en_salud_final['value'], errors='coerce')

#Ordenar filas por país, indicador y año
df_gastos_en_salud_final = df_gastos_en_salud_final.sort_values(by=['country_name', 'indicator_name', 'year']).reset_index(drop=True)

# Vista rápida
display(df_gastos_en_salud_final.head(100))



,country_name,country_code,indicator_name,year,value
0,Ingreso mediano,MIC,Gasto corriente en salud (% del PIB),2000,4.976600
1,Ingreso mediano,MIC,Gasto corriente en salud (% del PIB),2001,4.992908
2,Ingreso mediano,MIC,Gasto corriente en salud (% del PIB),2002,4.846495
3,Ingreso mediano,MIC,Gasto corriente en salud (% del PIB),2003,4.929510
4,Ingreso mediano,MIC,Gasto corriente en salud (% del PIB),2004,4.906570
...,...,...,...,...,...
95,América Latina y el Caribe,LCN,Gasto corriente en salud (% del PIB),2004,6.600314
96,América Latina y el Caribe,LCN,Gasto corriente en salud (% del PIB),2005,6.672393
97,América Latina y el Caribe,LCN,Gasto corriente en salud (% del PIB),2006,6.716483
98,América Latina y el Caribe,LCN,Gasto corriente en salud (% del PIB),2007,6.883672


In [277]:
#Ver si el dataframe tiene valores nulos
df_gastos_en_salud_final.isna().any()


,0
country_name,False
country_code,False
indicator_name,False
year,False
value,False


# 5. PREPARACION DE LA BASE INDICE DE POBREZA

In [278]:
display(df_indice_pobreza)
df_indice_pobreza

,region_name,region_code,country_name,country_code,reporting_year,reporting_level,survey_acronym,survey_coverage,survey_year,welfare_type,...,reporting_pop,reporting_gdp,reporting_pce,is_interpolated,distribution_type,estimation_type,spl,spr,pg,estimate_type
0,Sub-Saharan Africa,SSF,Angola,AGO,2000,national,HBS,national,2000.21,consumption,...,16310860.19,1932.988479,NaN,False,micro,survey,3.950,0.371469,9.609202,NaN
1,Sub-Saharan Africa,SSF,Angola,AGO,2008,national,IBEP-MICS,national,2008.50,consumption,...,21996714.00,3193.287723,1268.614278,False,micro,survey,3.967,0.353358,6.838756,NaN
2,Sub-Saharan Africa,SSF,Angola,AGO,2018,national,IDREA,national,2018.17,consumption,...,31480496.09,2775.746423,1605.110432,False,micro,survey,3.240,0.421619,10.964242,NaN
3,Europe & Central Asia,ECS,Albania,ALB,1996,national,EWS,national,1996.00,consumption,...,3168033.00,1683.769655,1522.871873,False,micro,survey,5.010,0.203902,4.209428,NaN
4,Europe & Central Asia,ECS,Albania,ALB,2002,national,LSMS,national,2002.00,consumption,...,3051010.00,2297.108535,1572.295460,False,micro,survey,4.859,0.224404,4.429310,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2551,Sub-Saharan Africa,SSF,Zambia,ZMB,2015,national,LCMS-VII,national,2015.00,consumption,...,16399089.00,1295.877887,603.812058,False,micro,survey,3.000,0.678928,22.894125,NaN
2552,Sub-Saharan Africa,SSF,Zambia,ZMB,2022,national,LCMS-VIII,national,2022.00,consumption,...,20152938.00,1298.848050,NaN,False,imputed,survey,3.000,0.716561,22.853020,NaN
2553,Sub-Saharan Africa,SSF,Zimbabwe,ZWE,2011,national,ICES,national,2011.00,consumption,...,13595424.00,1184.329610,884.907206,False,group,survey,3.266,0.401180,8.509818,NaN
2554,Sub-Saharan Africa,SSF,Zimbabwe,ZWE,2017,national,PICES,national,2017.00,consumption,...,14812482.00,1422.193460,1166.657928,False,micro,survey,3.000,0.446569,9.633754,NaN


,region_name,region_code,country_name,country_code,reporting_year,reporting_level,survey_acronym,survey_coverage,survey_year,welfare_type,...,reporting_pop,reporting_gdp,reporting_pce,is_interpolated,distribution_type,estimation_type,spl,spr,pg,estimate_type
0,Sub-Saharan Africa,SSF,Angola,AGO,2000,national,HBS,national,2000.21,consumption,...,16310860.19,1932.988479,NaN,False,micro,survey,3.950,0.371469,9.609202,NaN
1,Sub-Saharan Africa,SSF,Angola,AGO,2008,national,IBEP-MICS,national,2008.50,consumption,...,21996714.00,3193.287723,1268.614278,False,micro,survey,3.967,0.353358,6.838756,NaN
2,Sub-Saharan Africa,SSF,Angola,AGO,2018,national,IDREA,national,2018.17,consumption,...,31480496.09,2775.746423,1605.110432,False,micro,survey,3.240,0.421619,10.964242,NaN
3,Europe & Central Asia,ECS,Albania,ALB,1996,national,EWS,national,1996.00,consumption,...,3168033.00,1683.769655,1522.871873,False,micro,survey,5.010,0.203902,4.209428,NaN
4,Europe & Central Asia,ECS,Albania,ALB,2002,national,LSMS,national,2002.00,consumption,...,3051010.00,2297.108535,1572.295460,False,micro,survey,4.859,0.224404,4.429310,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2551,Sub-Saharan Africa,SSF,Zambia,ZMB,2015,national,LCMS-VII,national,2015.00,consumption,...,16399089.00,1295.877887,603.812058,False,micro,survey,3.000,0.678928,22.894125,NaN
2552,Sub-Saharan Africa,SSF,Zambia,ZMB,2022,national,LCMS-VIII,national,2022.00,consumption,...,20152938.00,1298.848050,NaN,False,imputed,survey,3.000,0.716561,22.853020,NaN
2553,Sub-Saharan Africa,SSF,Zimbabwe,ZWE,2011,national,ICES,national,2011.00,consumption,...,13595424.00,1184.329610,884.907206,False,group,survey,3.266,0.401180,8.509818,NaN
2554,Sub-Saharan Africa,SSF,Zimbabwe,ZWE,2017,national,PICES,national,2017.00,consumption,...,14812482.00,1422.193460,1166.657928,False,micro,survey,3.000,0.446569,9.633754,NaN


In [279]:
df_indice_pobreza.isna().sum() #La base posee varios NA's

,0
region_name,0
region_code,0
country_name,0
country_code,0
reporting_year,0
reporting_level,0
survey_acronym,0
survey_coverage,0
survey_year,0
welfare_type,0


In [280]:
df_indice_pobreza["ppp"]

,ppp
0,203.688736
1,203.688736
2,203.688736
3,50.772305
4,50.772305
...,...
2551,6.189585
2552,6.189585
2553,0.644238
2554,0.644238


In [281]:
df_indice_pobreza["ppp"].max()

15517657088.0

In [282]:
df_indice_pobreza["ppp"].min()

0.344110966

In [283]:
mediana_ppp = df_indice_pobreza["ppp"].median() # Utilizamos el valor de la mediana para rellenar valores faltantes

In [284]:
df_indice_pobreza["watts"] # Watts es una de las columnas que posee NA's
df_indice_pobreza["watts"].max()

1.284744380883

In [285]:
df_indice_pobreza["watts"].min()

0.0

In [286]:
mediana_watts =  df_indice_pobreza["watts"].median()
print(mediana_watts) # Utilizaremos la mediana para estimar rellenar los valores faltantes

0.010318561138


In [287]:
df_indice_pobreza["reporting_pce"] #reporting_pce posee NA's

,reporting_pce
0,NaN
1,1268.614278
2,1605.110432
3,1522.871873
4,1572.295460
...,...
2551,603.812058
2552,NaN
2553,884.907206
2554,1166.657928


In [288]:
df_indice_pobreza["reporting_pce"].max()

45129.1621450025

In [289]:
df_indice_pobreza["reporting_pce"].min()

80.392231912558

In [290]:
mediana_pce = df_indice_pobreza["reporting_pce"].median() #Utilizaremos la mediana para estimar los valores faltantes

In [291]:
df_indice_pobreza["watts"] = df_indice_pobreza["watts"].fillna(df_indice_pobreza["watts"].median())
df_indice_pobreza["ppp"] = df_indice_pobreza["ppp"].fillna(df_indice_pobreza["ppp"].median())
df_indice_pobreza["reporting_pce"] = df_indice_pobreza["reporting_pce"].fillna(df_indice_pobreza["reporting_pce"].median())



In [292]:
# Por otra parte, la columna "estimate_type" está completamente llena de NA's, por lo que no aporta nada de valor al análisis y es mejor eliminarla completamente.
print(df_indice_pobreza.columns)
df_indice_pobreza
df_indice_pobreza= df_indice_pobreza.drop(columns=["estimation_type"])


Index(['region_name', 'region_code', 'country_name', 'country_code',
       'reporting_year', 'reporting_level', 'survey_acronym',
       'survey_coverage', 'survey_year', 'welfare_type',
       'survey_comparability', 'comparable_spell', 'poverty_line', 'headcount',
       'poverty_gap', 'poverty_severity', 'watts', 'mean', 'median', 'mld',
       'gini', 'polarization', 'decile1', 'decile2', 'decile3', 'decile4',
       'decile5', 'decile6', 'decile7', 'decile8', 'decile9', 'decile10',
       'cpi', 'ppp', 'reporting_pop', 'reporting_gdp', 'reporting_pce',
       'is_interpolated', 'distribution_type', 'estimation_type', 'spl', 'spr',
       'pg', 'estimate_type'],
      dtype='object')


In [293]:
df_indice_pobreza.isna().sum() # Ahora verificamos que el data frame no posee NA's

,0
region_name,0
region_code,0
country_name,0
country_code,0
reporting_year,0
reporting_level,0
survey_acronym,0
survey_coverage,0
survey_year,0
welfare_type,0


In [294]:
df_indice_pobreza.dtypes #Los variables son del tipo que se espera de ellas

,0
region_name,object
region_code,object
country_name,object
country_code,object
reporting_year,int64
reporting_level,object
survey_acronym,object
survey_coverage,object
survey_year,float64
welfare_type,object


In [295]:
df_indice_pobreza.duplicated().sum() # No tenemos columnas duplicadas

np.int64(0)

In [296]:
df_indice_pobreza.head(10)

,region_name,region_code,country_name,country_code,reporting_year,reporting_level,survey_acronym,survey_coverage,survey_year,welfare_type,...,ppp,reporting_pop,reporting_gdp,reporting_pce,is_interpolated,distribution_type,spl,spr,pg,estimate_type
0,Sub-Saharan Africa,SSF,Angola,AGO,2000,national,HBS,national,2000.21,consumption,...,203.688736,16310860.19,1932.988479,5102.096719,False,micro,3.950,0.371469,9.609202,NaN
1,Sub-Saharan Africa,SSF,Angola,AGO,2008,national,IBEP-MICS,national,2008.50,consumption,...,203.688736,21996714.00,3193.287723,1268.614278,False,micro,3.967,0.353358,6.838756,NaN
2,Sub-Saharan Africa,SSF,Angola,AGO,2018,national,IDREA,national,2018.17,consumption,...,203.688736,31480496.09,2775.746423,1605.110432,False,micro,3.240,0.421619,10.964242,NaN
3,Europe & Central Asia,ECS,Albania,ALB,1996,national,EWS,national,1996.00,consumption,...,50.772305,3168033.00,1683.769655,1522.871873,False,micro,5.010,0.203902,4.209428,NaN
4,Europe & Central Asia,ECS,Albania,ALB,2002,national,LSMS,national,2002.00,consumption,...,50.772305,3051010.00,2297.108535,1572.295460,False,micro,4.859,0.224404,4.429310,NaN
5,Europe & Central Asia,ECS,Albania,ALB,2005,national,LSMS,national,2005.00,consumption,...,50.772305,3011487.00,2712.869838,1969.241694,False,micro,5.450,0.209975,3.860997,NaN
6,Europe & Central Asia,ECS,Albania,ALB,2008,national,LSMS,national,2008.00,consumption,...,50.772305,2947314.00,3345.981618,2735.224128,False,micro,5.770,0.172610,3.450628,NaN
7,Europe & Central Asia,ECS,Albania,ALB,2012,national,LSMS,national,2012.00,consumption,...,50.772305,2900401.00,3720.228765,2875.740716,False,micro,5.684,0.189182,3.634575,NaN
8,Europe & Central Asia,ECS,Albania,ALB,2014,national,HBS,national,2014.00,consumption,...,50.772305,2889104.00,3883.632628,3030.085146,False,micro,5.714,0.254353,3.839190,NaN
9,Europe & Central Asia,ECS,Albania,ALB,2015,national,HBS,national,2015.00,consumption,...,50.772305,2880703.00,3981.726623,3090.925034,False,micro,6.644,0.202021,3.029822,NaN


# 6. Preparacion de la base HIV

In [297]:
df_HIV_subset = df_HIV.iloc[:, [0, 1, 2] + list(range(78, 83))]  #del data frame solo conservar estas columnas

df_HIV_subset.columns =['Anio', 'codigo', 'pais', 'todas_edades','ninos', 'mujeres', 'hombres', 'adultos'] #asignar un nombre


# Verificar
print(df_HIV_subset.head(5))
print(df_HIV_subset.dtypes)
#df_HIV_subset=df_HIV_subset.apply(pd.to_numeric, errors='coerce') #convertir a tipo numerico
#Verificar por NAs


# Lista de columnas que quieres convertir
cols_numericas = ['todas_edades', 'ninos', 'mujeres', 'hombres', 'adultos']

# Convertir esas columnas a numéricas (coerce convierte errores en NaN)
df_HIV_subset[cols_numericas] = df_HIV_subset[cols_numericas].apply(pd.to_numeric, errors='coerce')

print(df_HIV_subset.dtypes)
df_HIV_subset.isna().sum()
#al ser datos epidemiologicos no manipular NAs


   Anio codigo                  pais todas_edades ninos mujeres hombres  \
0  2010  UNAAP  Asia and the Pacific          ...   ...     ...     ...   
1  2010    AFG           Afghanistan          NaN   NaN     NaN     NaN   
2  2010    AUS             Australia        17347    63    1821   15463   
3  2010    BGD            Bangladesh          NaN    24     NaN     NaN   
4  2010    BTN                Bhutan          146     2      68      76   

  adultos  
0     ...  
1     NaN  
2   17284  
3     NaN  
4     144  
Anio             int64
codigo          object
pais            object
todas_edades    object
ninos           object
mujeres         object
hombres         object
adultos         object
dtype: object
Anio              int64
codigo           object
pais             object
todas_edades    float64
ninos           float64
mujeres         float64
hombres         float64
adultos         float64
dtype: object


/tmp/ipython-input-194717041.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_HIV_subset[cols_numericas] = df_HIV_subset[cols_numericas].apply(pd.to_numeric, errors='coerce')


,0
Anio,0
codigo,0
pais,0
todas_edades,963
ninos,1145
mujeres,988
hombres,991
adultos,977


In [298]:
#PRUEBA DE FUNCIONAIENTO DE LA BASE
df_cr = df_HIV_subset[df_HIV_subset['codigo'] == 'CRI']
df_cr.head(5)


# Agrupar por año y sumar los casos (o cualquier otra agregación)
df_cr_grouped = df_cr.groupby('Anio')[['todas_edades', 'ninos', 'mujeres', 'hombres', 'adultos']].sum().reset_index()

# Verificar
print(df_cr_grouped)


    Anio  todas_edades  ninos  mujeres  hombres  adultos
0   2010        4195.0   69.0    670.0   3456.0   4126.0
1   2011        4240.0   70.0    726.0   3444.0   4170.0
2   2012        4840.0   71.0    772.0   3998.0   4770.0
3   2013        5646.0   72.0    871.0   4703.0   5574.0
4   2014        6437.0   72.0    875.0   5490.0   6365.0
5   2015        7473.0   75.0   1070.0   6329.0   7399.0
6   2016        8137.0   75.0   1115.0   6947.0   8062.0
7   2017        9075.0   79.0   1211.0   7785.0   8996.0
8   2018       10005.0   48.0   1339.0   8619.0   9958.0
9   2019       10710.0   50.0   1312.0   9348.0  10661.0
10  2020       11244.0   48.0   1555.0   9641.0  11196.0
11  2021       12631.0   45.0   1607.0  10979.0  12586.0
12  2022       13051.0   40.0   1730.0  11281.0  13011.0
13  2023       13992.0   93.0   1926.0  11974.0  13900.0
14  2024       14695.0   49.0   1994.0  12653.0  14647.0


# 7 UNIFICACIÓN DE LAS BASES

In [299]:
#REVISION FINAL DE LOS DATA FRAMES

#df_dengue_final.head(5)
#df_tuberculosis.head(5)
#df_gasto_salud.head(6)
#df_indice_pobreza.head(5)
df_HIV_subset.head(5)


,Anio,codigo,pais,todas_edades,ninos,mujeres,hombres,adultos
0,2010,UNAAP,Asia and the Pacific,NaN,NaN,NaN,NaN,NaN
1,2010,AFG,Afghanistan,NaN,NaN,NaN,NaN,NaN
2,2010,AUS,Australia,17347.0,63.0,1821.0,15463.0,17284.0
3,2010,BGD,Bangladesh,NaN,24.0,NaN,NaN,NaN
4,2010,BTN,Bhutan,146.0,2.0,68.0,76.0,144.0


In [300]:

# Renombrar columnas para consistencia
df_tuberculosis.rename(columns={
    'Entity': 'country',
    'Year': 'year',
    'Estimated number of new cases of all forms of tuberculosis': 'tb_cases'
}, inplace=True)

df_HIV_subset.rename(columns={
    'Anio': 'year',
    'pais': 'country',
    'todas_edades': 'hiv_cases'
}, inplace=True)



# Merge progresivo por country y year
df_merge = df_dengue2_final.merge(df_tuberculosis[['country', 'year', 'tb_cases']],
                                 on=['country', 'year'], how='outer')

df_merge = df_merge.merge(df_HIV_subset[['country', 'year', 'hiv_cases']],
                           on=['country', 'year'], how='outer')

# Ordenar
df_merge.sort_values(['country', 'year'], inplace=True)

print(df_merge.head())


# Verificar
print(df_merge.head(20))



df_merge.to_excel("dengue2.xlsx", index=False)



/tmp/ipython-input-3064447063.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_HIV_subset.rename(columns={


       country code  year  dengue_cases  tb_cases  hiv_cases
0  AFGHANISTAN  AFG  2021         734.0       NaN        NaN
1  AFGHANISTAN  AFG  2022        1341.0       NaN        NaN
2  AFGHANISTAN  AFG  2023        1481.0       NaN        NaN
3  AFGHANISTAN  AFG  2024        4850.0       NaN        NaN
4  AFGHANISTAN  AFG  2025         115.0       NaN        NaN
           country code  year  dengue_cases  tb_cases  hiv_cases
0      AFGHANISTAN  AFG  2021         734.0       NaN        NaN
1      AFGHANISTAN  AFG  2022        1341.0       NaN        NaN
2      AFGHANISTAN  AFG  2023        1481.0       NaN        NaN
3      AFGHANISTAN  AFG  2024        4850.0       NaN        NaN
4      AFGHANISTAN  AFG  2025         115.0       NaN        NaN
5   AMERICAN SAMOA  ASM  1955           0.0       NaN        NaN
6   AMERICAN SAMOA  ASM  1979           0.0       NaN        NaN
7   AMERICAN SAMOA  ASM  1980           1.0       NaN        NaN
8   AMERICAN SAMOA  ASM  1981           1.0      

In [301]:
#merge de las bases con datos economicos


# Seleccionar solo las columnas deseadas
df_indice_pobreza = df_indice_pobreza[['country_name', 'country_code', 'reporting_year', 'ppp']]

# Verificar resultado
print(df_indice_pobreza.head())


# Renombrar columnas
df_indice_pobreza.rename(columns={
    'country_name': 'country',
    'country_code': 'code',
    'reporting_year': 'year',
    'ppp': 'ppp'
}, inplace=True)

# Verificar
print(df_indice_pobreza.head())





  country_name country_code  reporting_year         ppp
0       Angola          AGO            2000  203.688736
1       Angola          AGO            2008  203.688736
2       Angola          AGO            2018  203.688736
3      Albania          ALB            1996   50.772305
4      Albania          ALB            2002   50.772305
   country code  year         ppp
0   Angola  AGO  2000  203.688736
1   Angola  AGO  2008  203.688736
2   Angola  AGO  2018  203.688736
3  Albania  ALB  1996   50.772305
4  Albania  ALB  2002   50.772305


In [302]:

# Seleccionar solo las columnas deseadas
df_gastos_en_salud_final = df_gastos_en_salud_final[['country_name', 'country_code', 'year', 'value']]

# Renombrar para consistencia
df_gastos_en_salud_final.rename(columns={
    'country_name': 'country',
    'country_code': 'code',
    'year': 'year',
    'value': 'value'
}, inplace=True)

# Verificar
print(df_gastos_en_salud_final.head())


            country code  year     value
0   Ingreso mediano  MIC  2000  4.976600
1   Ingreso mediano  MIC  2001  4.992908
2   Ingreso mediano  MIC  2002  4.846495
3   Ingreso mediano  MIC  2003  4.929510
4   Ingreso mediano  MIC  2004  4.906570


In [303]:

# Normalizar nombres de país para evitar problemas
df_gastos_en_salud_final['country'] = df_gastos_en_salud_final['country'].str.strip().str.upper()
df_indice_pobreza['country'] = df_indice_pobreza['country'].str.strip().str.upper()

# Merge por country y year
df_economicos = df_gastos_en_salud_final.merge(
    df_indice_pobreza[['country', 'code', 'year', 'ppp']],
    on=['country', 'year'],
    how='outer'  # puedes usar 'inner' si solo quieres coincidencias exactas
)

# Verificar resultado
print(df_economicos.head(20))


       country code_x  year      value code_y  ppp
0   AFGANISTÁN    AFG  2002   9.443391    NaN  NaN
1   AFGANISTÁN    AFG  2003   8.941258    NaN  NaN
2   AFGANISTÁN    AFG  2004   9.808474    NaN  NaN
3   AFGANISTÁN    AFG  2005   9.948289    NaN  NaN
4   AFGANISTÁN    AFG  2006  10.622766    NaN  NaN
5   AFGANISTÁN    AFG  2007   9.904675    NaN  NaN
6   AFGANISTÁN    AFG  2008  10.256495    NaN  NaN
7   AFGANISTÁN    AFG  2009   9.818487    NaN  NaN
8   AFGANISTÁN    AFG  2010   8.569672    NaN  NaN
9   AFGANISTÁN    AFG  2011   8.561908    NaN  NaN
10  AFGANISTÁN    AFG  2012   7.897169    NaN  NaN
11  AFGANISTÁN    AFG  2013   8.805964    NaN  NaN
12  AFGANISTÁN    AFG  2014   9.528878    NaN  NaN
13  AFGANISTÁN    AFG  2015  10.105348    NaN  NaN
14  AFGANISTÁN    AFG  2016  11.818590    NaN  NaN
15  AFGANISTÁN    AFG  2017  12.620817    NaN  NaN
16  AFGANISTÁN    AFG  2018  14.208419    NaN  NaN
17  AFGANISTÁN    AFG  2019  14.831320    NaN  NaN
18  AFGANISTÁN    AFG  2020  15

In [306]:

# Consolidar df_merge (sumar casos numéricos)
df_merge = df_merge.groupby(['country', 'year'], as_index=False).sum(numeric_only=True)

# Consolidar df_economicos (promedio o primer valor)
df_economicos = df_economicos.groupby(['country', 'year'], as_index=False).mean(numeric_only=True)

# Merge final
df_unificado = df_merge.merge(df_economicos, on=['country', 'year'], how='outer')

# Filtrar años entre 2010 y 2024
df_unificado = df_unificado[(df_unificado['year'] >= 2010) & (df_unificado['year'] <= 2024)]

# Ordenar y exportar
df_unificado.sort_values(['country', 'year'], inplace=True)


df_unificado.head(10)



,country,year,dengue_cases,tb_cases,hiv_cases,value,ppp
8,AFGANISTÁN,2010,NaN,NaN,NaN,8.569672,NaN
9,AFGANISTÁN,2011,NaN,NaN,NaN,8.561908,NaN
10,AFGANISTÁN,2012,NaN,NaN,NaN,7.897169,NaN
11,AFGANISTÁN,2013,NaN,NaN,NaN,8.805964,NaN
12,AFGANISTÁN,2014,NaN,NaN,NaN,9.528878,NaN
13,AFGANISTÁN,2015,NaN,NaN,NaN,10.105348,NaN
14,AFGANISTÁN,2016,NaN,NaN,NaN,11.818590,NaN
15,AFGANISTÁN,2017,NaN,NaN,NaN,12.620817,NaN
16,AFGANISTÁN,2018,NaN,NaN,NaN,14.208419,NaN
17,AFGANISTÁN,2019,NaN,NaN,NaN,14.831320,NaN
